# Make sess class from raw data for each session and save as pickle

Run after suite2p curation but before anything else.

Currently uses info from sessions_dict.py to loop through sessions and create the sess class, \
synchronizing neural data with behavioral data.

sess pickle files will be named `<scene>_<session>_<scan>.pickle`  \
and saved in `path_dict['preprocessed_root']/sess/<animal>/<date>`.

Set `overwrite` to `True` if you want to overwrite existing .pickle files. Otherwise, you will get an error that the file already exists.

In [28]:
overwrite = True

In [29]:
import os
import numpy as np

from reward_relative import preprocessing as pp
from reward_relative import utilities as ut

import TwoPUtils


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Specify your path dictionary here.

Copy and rename `path_dict.py` to a new file and edit it with the paths on your system.

In [30]:
from reward_relative.path_dict_msosa_mac import path_dictionary as path_dict
path_dict

{'preprocessed_root': '/Users/marielenasosa/Data/SosaLab',
 'sbx_root': '/media/marielenasosa/T7/2p_raw_data',
 'gdrive_root': '/mnt/gdrive/2P_Data',
 'VR_Data': '/Users/marielenasosa/Data/SosaLab/VR_Data',
 'git_repo_root': '/Users/marielenasosa/gitrepos',
 'TwoPUtils': '/Users/marielenasosa/gitrepos/TwoPUtils',
 'home': '/Users/marielenasosa',
 'fig_dir': '/Users/marielenasosa/Data/SosaLab/fig_scratch'}

## Scroll or click to the desired section for:

[Behavior only](#Behavior-only)


Within each section, define animal and iterate through sessions.

While running the below cells, if you get an error that says `DatabaseError: Execution failed on sql 'SELECT * FROM data': no such table: data`,
check that all of your .sqlite files are named properly and have data in them (i.e. `Scene_1.sqlite` instead of `'Scene_1(1).sqlite'`


# Behavior only

In [31]:
from reward_relative.sessions_dict_behavior_only import sosalab as metadata

In [32]:
metadata

{'pp01': ({'date': '2026_06_03',
   'scene': 'FiveTower_Stay',
   'session': 1,
   'scan': nan,
   'exp_day': 1,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr',
   'reward_0': 'A',
   'reward_1': 'A'},
  {'date': '2026_06_04',
   'scene': 'FiveTower_Stay',
   'session': 1,
   'scan': nan,
   'exp_day': 2,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr',
   'reward_0': 'A',
   'reward_1': 'A'},
  {'date': '2026_06_05',
   'scene': 'FiveTower_Switch_BlackoutDelay',
   'session': 2,
   'scan': nan,
   'exp_day': 3,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr',
   'reward_0': 'A',
   'reward_1': 'B'},
  {'date': '2026_06_06',
   'scene': 'FiveTower_Stay_BlackoutDelay',
   'session': 1,
   'scan': nan,
   'exp_day': 4,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr',
   'reward_0': 'B',
   'reward_1': 'B'}),
 'pp03': (),
 'ps04': ()}

In [33]:
## Define animal
animal = 'pp01'
days = np.arange(0, len(metadata[animal])) # range of days
# days =days[1:3] # optional select subset of days
days

array([0, 1, 2, 3])

### Main cell to create sess

In [39]:
basedir = os.path.join(path_dict['preprocessed_root'], animal)
sbxdir = os.path.join(path_dict['sbx_root'], animal)
vrdir = path_dict['VR_Data']

binary_from_sbxdir = False # only relevant for downsampling
calcium_exists = False

load_suite2p = False
load_scaninfo = False
VR_only = True

trial_matrix_kwargs = []

for i, day in enumerate(days):

    if type(metadata[animal][day]) is not tuple:
        date = metadata[animal][day]['date']
        scene = metadata[animal][day]['scene']
        rig = metadata[animal][day]['rig']
        session = metadata[animal][day]['session']
        scan_number = metadata[animal][day]['scan']

        sess = pp.create_sess(basedir, sbxdir, vrdir, animal, date, rig, scene, session, scan_number,
                              load_scaninfo=load_scaninfo,
                              load_VR=True,
                              load_suite2p=load_suite2p,
                              load_behavior=True,
                              VR_only=VR_only,                              
                              )

        sess_dir = os.path.join(
            path_dict['preprocessed_root'], 'sess', animal, date)
        os.makedirs(sess_dir, exist_ok=True)
        print(sess_dir)

        if np.isnan(scan_number):
            scan_number=0
            
        sess_name = '%s_%03d_%03d.pickle' % (scene,
                                             session,
                                             scan_number
                                             )
        # Write sess to pickle file
        ut.write_sess_pickle(sess, sess_dir, sess_name, overwrite=overwrite)

    else:
        print("Iterating through multiple sessions")
        for i in range(len(metadata[animal][day])):
            date = metadata[animal][day][i]['date']
            scene = metadata[animal][day][i]['scene']
            session = metadata[animal][day][i]['session']
            scan_number = metadata[animal][day][i]['scan']

            sess = pp.create_sess(basedir, sbxdir, vrdir, animal, date, scene, session, scan_number,
                                  load_scaninfo=True,
                                  load_VR=True,
                                  load_suite2p=True,
                                  load_behavior=True)

            sess_dir = os.path.join(
                path_dict['preprocessed_root'], 'sess', animal, date)
            os.makedirs(sess_dir, exist_ok=True)
            print(sess_dir)

            sess_name = '%s_%03d_%03d.pickle' % (scene,
                                                 session,
                                                 scan_number,
                                                 )
            # Write sess to pickle file
            ut.write_sess_pickle(
                sess, sess_dir, sess_name, overwrite=overwrite)

/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_03/omen-vr/FiveTower_Stay_1.sqlite
/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_03/omen-vr/FiveTower_Stay_1.sqlite
Fixing teleports
(118264, 16)
/Users/marielenasosa/Data/SosaLab/sess/pp01/2026_06_03
writing FiveTower_Stay_001_000.pickle


/Users/marielenasosa/gitrepos/SosaLab/Sosa_et_al_2024/src/reward_relative/preprocessing.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if suite2p:


/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_04/omen-vr/FiveTower_Stay_1.sqlite
/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_04/omen-vr/FiveTower_Stay_1.sqlite
Fixing teleports
(109053, 16)


/Users/marielenasosa/gitrepos/SosaLab/Sosa_et_al_2024/src/reward_relative/preprocessing.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if suite2p:


/Users/marielenasosa/Data/SosaLab/sess/pp01/2026_06_04
writing FiveTower_Stay_001_000.pickle
/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_05/omen-vr/FiveTower_Switch_BlackoutDelay_2.sqlite
/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_05/omen-vr/FiveTower_Switch_BlackoutDelay_2.sqlite
Fixing teleports
(133273, 16)


/Users/marielenasosa/gitrepos/SosaLab/Sosa_et_al_2024/src/reward_relative/preprocessing.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if suite2p:


/Users/marielenasosa/Data/SosaLab/sess/pp01/2026_06_05
writing FiveTower_Switch_BlackoutDelay_002_000.pickle
/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_06/omen-vr/FiveTower_Stay_BlackoutDelay_1.sqlite
/Users/marielenasosa/Data/SosaLab/VR_Data/pp01/2026_06_06/omen-vr/FiveTower_Stay_BlackoutDelay_1.sqlite
Fixing teleports
(136587, 17)


/Users/marielenasosa/gitrepos/SosaLab/Sosa_et_al_2024/src/reward_relative/preprocessing.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if suite2p:


/Users/marielenasosa/Data/SosaLab/sess/pp01/2026_06_06
writing FiveTower_Stay_BlackoutDelay_001_000.pickle


In [40]:
sess = ut.load_sess_pickle(path_dict['preprocessed_root'], 'pp01', exp_day=1)

/Users/marielenasosa/Data/SosaLab/sess/pp01/2026_06_03/FiveTower_Stay_001_000.pickle


In [ ]:
sess.timeseries.keys()

array([[ 0.        ,  0.        ,  0.        , ..., 43.8017251 ,
        70.589379  , 22.50766475]])

# STOP HERE

# Single plane sessions

In [4]:
from reward_relative.sessions_dict import single_plane

In [6]:
## Define animal
animal = 'GCAMP15'
days = np.arange(0, len(single_plane[animal])) # range of days
days =days[1:3]
days

array([1, 2])

In [38]:
basedir = os.path.join(path_dict['preprocessed_root'], animal)
sbxdir = os.path.join(path_dict['sbx_root'], animal)
vrdir = path_dict['VR_Data']

binary_from_sbxdir = True # only relevant for downsampling
calcium_exists = True

load_suite2p = True
load_scaninfo = True
VR_only = False

trial_matrix_kwargs = []

for i, day in enumerate(days):

    if type(single_plane[animal][day]) is not tuple:
        date = single_plane[animal][day]['date']
        scene = single_plane[animal][day]['scene']
        session = single_plane[animal][day]['session']
        scan_number = single_plane[animal][day]['scan']

        sess = pp.create_sess(basedir, sbxdir, vrdir, animal, date, scene, session, scan_number,
                              load_scaninfo=load_scaninfo,
                              load_VR=True,
                              load_suite2p=load_suite2p,
                              load_behavior=True,
                              VR_only=VR_only,                              
                              )

        sess_dir = os.path.join(
            path_dict['preprocessed_root'], 'sess', animal, date)
        os.makedirs(sess_dir, exist_ok=True)
        print(sess_dir)

        if np.isnan(scan_number):
            scan_number=0
            
        sess_name = '%s_%03d_%03d.pickle' % (scene,
                                             session,
                                             scan_number
                                             )
        # Write sess to pickle file
        ut.write_sess_pickle(sess, sess_dir, sess_name, overwrite=overwrite)

    else:
        print("Iterating through multiple sessions")
        for i in range(len(single_plane[animal][day])):
            date = single_plane[animal][day][i]['date']
            scene = single_plane[animal][day][i]['scene']
            session = single_plane[animal][day][i]['session']
            scan_number = single_plane[animal][day][i]['scan']

            sess = pp.create_sess(basedir, sbxdir, vrdir, animal, date, scene, session, scan_number,
                                  load_scaninfo=True,
                                  load_VR=True,
                                  load_suite2p=True,
                                  load_behavior=True)

            sess_dir = os.path.join(
                path_dict['preprocessed_root'], 'sess', animal, date)
            os.makedirs(sess_dir, exist_ok=True)
            print(sess_dir)

            sess_name = '%s_%03d_%03d.pickle' % (scene,
                                                 session,
                                                 scan_number,
                                                 )
            # Write sess to pickle file
            ut.write_sess_pickle(
                sess, sess_dir, sess_name, overwrite=overwrite)

NameError: name 'single_plane' is not defined

# Multi plane sessions

In [12]:
from reward_relative.sessions_dict import multi_plane
#multi_plane

In [15]:
animal = 'GCAMP18'
days = np.arange(0, len(multi_plane[animal])) # range of days
nplanes = 2
days = days[1:18] #[2:4]
days

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [ ]:
basedir = os.path.join(path_dict['preprocessed_root'],animal)
sbxdir = os.path.join(path_dict['gdrive_root'], animal) #os.path.join(path_dict['sbx_root'],animal) 
vrdir = path_dict['VR_Data']

# Get data binary from basedir or sbxdir?
binary_from_sbxdir = False
calcium_exists = True
add_suite2p = True

for day in days: 
    if type(multi_plane[animal][day]) is not tuple:   
        date = multi_plane[animal][day]['date']
        scene = multi_plane[animal][day]['scene']
        session = multi_plane[animal][day]['session']
        scan_number = multi_plane[animal][day]['scan']

        fullpath = os.path.join(basedir,date,scene,"%s_%03d_%03d" % (scene, session, scan_number))
        scanpath = os.path.join(sbxdir,date,scene,"%s_%03d_%03d" % (scene, session, scan_number)) #change back to sbxdir

        sess = pp.create_sess(basedir,sbxdir,vrdir,animal,date,scene,session,scan_number,
                               load_scaninfo=True,
                               load_VR = True,
                               load_suite2p = add_suite2p,
                               load_behavior = True)

        nframes = int(sess.scan_info['max_idx']/sess.n_planes)

        sess_dir = os.path.join(path_dict['preprocessed_root'],'sess',animal,date)

        os.makedirs(sess_dir,exist_ok=True)
        print(sess_dir)

        sess_name = '%s_%03d_%03d.pickle' % (scene, 
                                             session,
                                             scan_number,
                                             )
        # Write sess to pickle file
        ut.write_sess_pickle(sess,sess_dir,sess_name,overwrite=overwrite)

    else:
        print("Iterating through multiple sessions")
        for i in range(len(multi_plane[animal][day])):
            date = multi_plane[animal][day][i]['date']
            scene = multi_plane[animal][day][i]['scene']
            session = multi_plane[animal][day][i]['session']
            scan_number = multi_plane[animal][day][i]['scan']

            fullpath = os.path.join(basedir,date,scene,"%s_%03d_%03d" % (scene, session, scan_number))
            scanpath = os.path.join(sbxdir,date,scene,"%s_%03d_%03d" % (scene, session, scan_number))

            sess = pp.create_sess(basedir,sbxdir,vrdir,animal,date,scene,session,scan_number,
                       load_scaninfo=True,
                       load_VR = True,
                       load_suite2p = add_suite2p,
                       load_behavior = True)

            nframes = int(sess.scan_info['max_idx']/sess.n_planes)

            sess_dir = os.path.join(path_dict['preprocessed_root'],'sess',animal,multi_plane[animal][day][i]['date'])

            os.makedirs(sess_dir,exist_ok=True)
            print(sess_dir)

            sess_name = '%s_%03d_%03d.pickle' % (scene, 
                                                 session,
                                                 scan_number,
                                                 )
            # Write sess to pickle file
            ut.write_sess_pickle(sess,sess_dir,sess_name,overwrite=overwrite)
